# 📊 Exploración y Validación: Coupons, Categories e Inventory

Este notebook realiza el **análisis exploratorio de calidad de datos** y la **validación de limpieza (Antes vs Después)** para tres tablas clave del pipeline:
1. **`coupons`** (Cupones de descuento)
2. **`categories`** (Categorías de productos)
3. **`inventory`** (Inventario y existencias en almacenes)

---

## ⚙️ Celda 1: Inicialización de Spark y Conexión al Lakehouse (Bronze)

**¿Qué hace este código?**
- `try...except`: Inicializa `spark` de forma segura. Si estás en Databricks o VS Code con Databricks Connect, reutiliza la sesión existente o se conecta a tu cluster.
- `storage_account`: Nombre de tu cuenta de almacenamiento de Azure Blob Storage.
- `container`: Nombre del contenedor donde residen los datos crudos (`bronze`).
- `spark.read.option("multiLine", True).json(path)`: Lee los archivos JSON multilínea de cada entidad.


In [7]:
# 1. Inicialización universal de la sesión de Spark
try:
    spark
except NameError:
    try:
        from databricks.connect import DatabricksSession
        spark = DatabricksSession.builder.getOrCreate()
    except Exception:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.appName("ExploracionNuevasTablas").getOrCreate()

# 2. Configuración de rutas en Azure Data Lake Storage (Bronze)
storage_account = "stecommercedirty01"  # Tu cuenta de almacenamiento de Azure
container = "bronze"                    # Capa Bronze (datos crudos sin limpiar)

# 3. Lectura de las 3 tablas crudas desde Bronze
path_coupons = f"abfss://{container}@{storage_account}.dfs.core.windows.net/coupons/"
path_categories = f"abfss://{container}@{storage_account}.dfs.core.windows.net/categories/"
path_inventory = f"abfss://{container}@{storage_account}.dfs.core.windows.net/inventory/"

df_coupons_raw = spark.read.option("multiLine", True).json(path_coupons)
df_categories_raw = spark.read.option("multiLine", True).json(path_categories)
df_inventory_raw = spark.read.option("multiLine", True).json(path_inventory)

print(f"✅ Coupons crudos cargados:    {df_coupons_raw.count()} registros")
print(f"✅ Categories crudas cargadas: {df_categories_raw.count()} registros")
print(f"✅ Inventory crudo cargado:    {df_inventory_raw.count()} registros")


✅ Coupons crudos cargados:    100 registros
✅ Categories crudas cargadas: 24 registros
✅ Inventory crudo cargado:    1600 registros


--- 
## 🎟️ SECCIÓN 1: Exploración y Limpieza de `coupons`

### Problemas detectados en los datos crudos:
- `code`: Tiene códigos con espacios en blanco, minúsculas o vacíos.
- `discount_type`: Viene desordenado (`FIXED`, `dollar`, `fixed`, `PERCENTAGE`, `percent`, `bogo`, `invalid_type`).
- `is_active`: Mezcla `'1'`, `'0'`, `'TRUE'`, `'FALSE'`, `'yes'`, `'no'`.
- `discount_value`, `min_order`: Vienen como `string` en vez de números flotantes con decimales.
- `max_uses`, `used_count`: Vienen como `string` en vez de enteros.

In [8]:
# Diagnóstico rápido de tipos de descuento crudos
print("🔍 Valores únicos de discount_type en Bronze:")
df_coupons_raw.select("discount_type").distinct().show(truncate=False)

# Diagnóstico rápido de estados booleanos crudos
print("🔍 Valores únicos de is_active en Bronze:")
df_coupons_raw.select("is_active").distinct().show(truncate=False)


🔍 Valores únicos de discount_type en Bronze:
+---------------+
|discount_type  |
+---------------+
|dollar         |
|percentage     |
|bogo           |
|123            |
|fixed          |
|PERCENTAGE     |
|percent        |
|Percentage     |
|invalid_type   |
|NULL           |
|Fixed          |
|free_shipping  |
|FIXED          |
|buy_one_get_one|
|               |
+---------------+

🔍 Valores únicos de is_active en Bronze:
+---------+
|is_active|
+---------+
|false    |
|0        |
|TRUE     |
|yes      |
|true     |
|FALSE    |
|1        |
|no       |
|NULL     |
|         |
+---------+



In [9]:
# Ejecutar la limpieza de coupons y ver ANTES vs DESPUÉS
from transformacion.cleaning.coupon_cleaner import run_coupon_cleaning

df_coupons_clean = run_coupon_cleaning(df_coupons_raw, spark)

# Comparación Antes vs Después de las columnas clave
print("👀 Comparación Antes vs Después en Coupons:")
df_coupons_clean.select(
    "code", "code_clean",
    "discount_type", "discount_type_clean",
    "is_active", "expires_at", "expires_at_clean"
).show(20, truncate=False)


KeyboardInterrupt: 

--- 
## 🏷️ SECCIÓN 2: Exploración y Limpieza de `categories`

### Problemas detectados en los datos crudos:
- `name`: Caracteres especiales indebidos y espacios redundantes (debe conservar caracteres válidos como el ampersand `&` en 'Food & Beverages').
- `description`: Nulos o cadenas vacías que deben tener un valor informativo ('No description').
- `created_at`: Valores erróneos como `'bad-date'` o fechas futuras de prueba que requieren parseo estandarizado a ISO.

In [10]:
# Ejecutar la limpieza de categories y comparar
from transformacion.cleaning.category_cleaner import run_category_cleaning

df_categories_clean = run_category_cleaning(df_categories_raw, spark)

print("👀 Categorías limpias: Antes vs Después:")
df_categories_clean.select(
    "id",
    "name", "name_clean",
    "description", "description_clean",
    "is_active",
    "created_at", "created_at_clean"
).show(truncate=False)


ValueError: Modo 'none' no reconocido. Usa: 'title', 'upper' o 'lower'

--- 
## 📦 SECCIÓN 3: Exploración y Limpieza de `inventory`

### Problemas detectados en los datos crudos:
- `warehouse`: Nombres con espacios alrededor (`'  Main Hub  '`), inconsistencias en mayúsculas/minúsculas y valores vacíos.
- `status`: Valores con espacios (`'in stock'`), mayúsculas (`'LOW_STOCK'`), valores nulos o `'N/A'`.
- `last_restock`: Fechas con texto corrupto (`'bad-date'`) que deben parsearse a formato seguro.
- `quantity`, `reorder_point`: Llegan como strings en lugar de enteros.

In [ ]:
# Diagnóstico rápido de estados en inventario
print("🔍 Valores únicos de status en Inventory:")
df_inventory_raw.groupBy("status").count().show(truncate=False)


In [ ]:
# Ejecutar la limpieza de inventario y comparar
from transformacion.cleaning.inventory_cleaner import run_inventory_cleaning

df_inventory_clean = run_inventory_cleaning(df_inventory_raw, spark)

print("👀 Inventario Antes vs Después (Muestra de 25 registros):")
df_inventory_clean.select(
    "product_id",
    "warehouse", "warehouse_clean",
    "status", "status_clean",
    "quantity", "reorder_point",
    "last_restock", "last_restock_clean"
).show(25, truncate=False)


--- 
## 🚀 SECCIÓN 4: Pipeline Completo Integrado

Aquí ejecutamos `run_transformacion()`, el orquestador principal que procesa todas las entidades en un solo flujo unificado listo para la capa Silver.

In [ ]:
from transformacion.pipeline_transformacion import run_transformacion

# Empaquetamos todas las tablas crudas en un diccionario
dataframes_crudos = {
    "coupons": df_coupons_raw,
    "categories": df_categories_raw,
    "inventory": df_inventory_raw,
}

# Corremos el pipeline integrado
dataframes_limpios = run_transformacion(dataframes_crudos, spark)

print("🎉 Resumen del Pipeline Ejecutado:")
for entidad, df_limpio in dataframes_limpios.items():
    print(f"  • {entidad}: {df_limpio.count()} filas transformadas y auditadas")
